## PDF Extraction and Embedding (Programme Documents)

### 1. Extracting raw text data

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import os, re

In [2]:
eee_path = "https://www.polyu.edu.hk/eee/study/information-for-current-students/programme-documents/"
pdf_path = "../RAG/ProgramBooklet"

path = os.path.join(pdf_path)
pdfs = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
llm = ChatOllama(model="deepseek-r1:8b", validate_model_on_init=True, temperature=0.6, reasoning=False)
print(pdfs)

['MSc_EIE_46011_2526.pdf', 'BEngBSc_Scheme_IAIE_46409_2526.pdf', 'PhDMPhil_EEE_46601_2526.pdf', 'MSc_EE_46010_2526.pdf', 'BEng_Scheme_EE_46408_2526.pdf', 'MSc_EV_46012_2526.pdf', 'MSc_MQ_46013_2526.pdf']


In [6]:
def regex_enhance(txt):
    text = re.sub(r' {2,}', ' ', txt)  # Strip excess white spaces
    return text

def extract_programme_title(first_page_content, llm_model):
    """
    Extract programme title from first page using LLM.
    """
    # Take first 2000 characters to avoid token limits
    content_snippet = first_page_content[:2000]
    
    prompt = \
        f"""
        You are extracting the title from a university programme booklet's first page.

        Extract the full programme title including:
            1. Degree type (e.g., Bachelor of Engineering, Master of Science, PhD)
            2. Discipline (e.g., Electrical Engineering, Electronic and Information Engineering)

        Return only the programme title with normalized capitalization, nothing else!

        *First page content*:
        {content_snippet}

        Programme Title:
        """
    
    # Extract programme title from first page using LLM
    try:
        response = llm_model.invoke(prompt)
        title = response.content.strip()
        
        # Clean up the response
        title = re.sub(r'\s+', ' ', title)
        title = title.replace('"', '').replace("'", "")
        
        # Validate it's a reasonable title
        if len(title) > 10 and len(title) < 250:
            return title
        else:
            return "Unknown"
    except Exception as e:
        print(f"LLM extraction failed: {e}")
        return "Unknown"

docs = []
unwanted_metadata = ["producer", "creator", "creationdate", "file_path", 
                     "format", "title", "subject", "keywords", "moddate", 
                     "author", "trapped", "modDate", "creationDate"]

for pdf in pdfs:
    loader = PyMuPDFLoader(f"{pdf_path}/{pdf}")
    cur_pdf = loader.load()
    
    # Extract programme title from first page
    programme_title = ""
    if len(cur_pdf) > 0:
        programme_title = extract_programme_title(cur_pdf[0].page_content, llm)
        print(f"PDF: {pdf} -> Programme: {programme_title}")
    
    for doc in cur_pdf:
        # Augment metadata for PDFs
        doc.page_content = regex_enhance(doc.page_content)
        doc.metadata["source"] = pdf
        doc.metadata["content_type"] = "pdf"
        doc.metadata["programme_title"] = programme_title
        for key in unwanted_metadata:
            if key in doc.metadata:
                del doc.metadata[key]
    docs.extend(cur_pdf)

PDF: MSc_EIE_46011_2526.pdf -> Programme: Master Of Science In Electronic And Information Engineering
PDF: BEngBSc_Scheme_IAIE_46409_2526.pdf -> Programme: Bachelor of Engineering and Bachelor of Science in Information and Artificial Intelligence Engineering
PDF: PhDMPhil_EEE_46601_2526.pdf -> Programme: PhD / MPhil in Electrical and Electronic Engineering
PDF: MSc_EE_46010_2526.pdf -> Programme: Master Of Science In Electrical Engineering
PDF: BEng_Scheme_EE_46408_2526.pdf -> Programme: Bachelor of Engineering Hons Scheme in Electrical Engineering
PDF: MSc_EV_46012_2526.pdf -> Programme: Master Of Science In Electric Vehicles
PDF: MSc_MQ_46013_2526.pdf -> Programme: Master Of Science In Microelectronics And Quantum Systems Engineering


In [12]:
#print(f"Extracted number of documents in PDFs: {len(docs)}")
'''
for doc in docs:
    print(f"Source: {doc.metadata.get('source')}, type: {doc.metadata.get('content_type')}")
'''
print(docs[12].page_content)

11 
 
5.3 
Study Load 
 
For students following the progression pattern specified for their programme, they have 
to take the number of credits, as specified in the Programme Requirement Document, 
for each semester. 
 
The normal study load is 15 credits in a semester for full-time study. The maximum 
study load to be taken by a student in a semester is 21 credits, unless exceptional 
approval is given by the Head of the programme offering Department. For such cases, 
students should be reminded that the study load approved should not be taken as grounds 
for academic appeal. 
 
To help improve the academic performance of students on academic probation, these 
students will be required to take a reduced study load in the following semester (Summer 
Term excluded). The maximum number of credits to be taken by the students varies 
according to the policies of individual Departments and will be subject to the approval 
of the authorities concerned. 
 
Students who have obtained approval 

### 2. Text Splitting

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    page = chunk.metadata.get("page", "N/A")
    programme = chunk.metadata.get("programme_title", "N/A")
    chunk.metadata["chunk_id"] = f"PolyU_Doc_{source}_page_{page}_chunk_{i}"
    del chunk.metadata["programme_title"]
    
    chunk.page_content = f"--- Source: {source}, Programme: {programme} --- \n --- Retrieved from: {eee_path} --- \n\n{chunk.page_content}"

In [7]:
print(chunks[50])

page_content='completion of the late assessment. 
 
The student concerned is required to submit his/her application for late assessment in writing to the 
Head of Department offering the subject, within five working days from the date of the examination, 
together with any original supporting documents. Approval of applications for late assessment 
and the means for such late assessments shall be given by the Head of Department offering the 
subject or the subject teacher concerned, in consultation with the Programme Leader. Verification 
of the supporting documents with the issuing authority may be conducted by the subject offering 
Department as part of the approval process. 
 
5.11 Assessment to be competed 
 
For cases where students fail marginally in one of the components within a subject, the BoE can 
defer making a decision until the students concerned have completed the necessary remedial work 
to the satisfaction of the subject examiner(s). The remedial work must not take the

### 3. Document Embedding in Chroma

In [3]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "academic_documents" if not SINGLE else "vaa_documents"

In [4]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

client.get_collection(name=collection_name).count()

4291

In [10]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i + 1500)]
    )

print(f"Added {len(chunks)} chunks into ChromaDB")

Added 3179 chunks into ChromaDB


### 4. Simple Testing

In [ ]:
vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

query = "What is Master of Science of Electronic and Information Engineering?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content[:]}...")
    print(f"Source: {result.metadata.get('source')}, Page: {result.metadata.get('page')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

--- Source: MSc_EE_46010_2526.pdf, Programme: Master Of Science In Electrical Engineering --- 
 --- Retrieved from: https://www.polyu.edu.hk/eee/study/information-for-current-students/programme-documents/ --- 

Master of Science in Electrical Engineering 2025/26 
 
1 
 
1 
General Information 
 
1.1 
Programme Information 
 
Programme Title (Code) 
 
Master of Science in Electrical Engineering (46010) 
電機工程學理學碩士學位 
 
Host Department 
 
Department of Electrical and Electronic Engineering 
 
Mode of Study and Normal Duration 
 
Mode 
Normal Duration 
Mixed-Mode 
Full-time: 1.5 years (3 semesters) 
Part-time: 2.5 years (5 semesters) 
 
Students should complete the programme within the normal duration of the programme. Those 
who exceed the normal duration of the programme will be de-registered from the programme 
unless prior approval has been obtained from relevant authorities. 
 
Award Title 
 
Students will be awarded one of the following awards upon successful completion of the 
requi